In [8]:
import sys
import math
import re
import numpy as np
import pandas as pd
import dataframe_image as dfi
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter as letter
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.transforms import Bbox

sys.path.append('..')
from modules.source import PROCESSED_DATA_PATH, OUTPUT_FOLDER_PATH

analysis_dfs = pd.read_pickle(f"{PROCESSED_DATA_PATH}/analysis_dfs.pkl")

### Table as PNGs

In [9]:
df_b = analysis_dfs['Beta & Confidence Intervals']['df']
for d in df_b.columns.get_level_values(0).unique():
   w = df_b[(d, 'CI_upper')] - df_b[(d, 'CI_lower')]
   df_b[(d, 'CI Relative Width Ratio')] = w / df_b[(d, 'beta')].abs()

# Re-sort columns to keep metric ordering consistent (beta, CI_lower, CI_upper, CI Relative Width Ratio)
metrics = ['beta', 'CI_lower', 'CI_upper', 'CI Relative Width Ratio']
cols = pd.MultiIndex.from_product([df_b.columns.get_level_values(0).unique(), metrics])
analysis_dfs['Beta & Confidence Intervals']['df'] = df_b.reindex(columns=cols)

In [10]:
out = f'{OUTPUT_FOLDER_PATH}/analysis_tables/'
fmts = {'ADF VIF Diagnostics': {('ADF', 'score'): '{:.3f}', ('ADF', 'p-val'): '{:.2e}', ('VIF', 'score'): '{:.2f}'}, 'Variance Decomposition': '{:.2f}', 'Beta & Confidence Intervals': '{:.3g}'}

def save(df, title, fmt='{:.3f}', name='', shade=None, bold=None):
   n, lv = df.index.nlevels, df.index.to_frame(False).astype(str)
   lv = lv.mask((lv == lv.shift()).cumprod(axis=1).astype(bool), '')
   f = [(fmt.get(c, fmt.get(c[1] if isinstance(c, tuple) and len(c) > 1 else c, '{:.3f}')) if isinstance(fmt, dict) else fmt) for c in df.columns]
   body = [[s.format(v) for v, s in zip(r, f)] for r in df.to_numpy()]
   head = ['\n'.join(map(str, c)) if isinstance(c, tuple) else str(c) for c in df.columns]
   rows = [list(df.index.names) + head] + [list(a) + b for a, b in zip(lv.to_numpy(), body)]
   fig, ax = plt.subplots(figsize=(30, 20))
   ax.axis('off')
   t = ax.table(rows, cellLoc='center', loc='upper left')
   t.auto_set_font_size(False); t.set_fontsize(10); t.auto_set_column_width(range(len(rows[0]))); t.scale(1, 1.8)
   hm, (m, fc, tc) = 1 + max(h.count('\n') for h in head), shade if shade else (None, None, None)
   for (i, j), c in t.get_celld().items():
      if i == 0:
         c.set_facecolor('#f3f4f6'); c.set_text_props(fontweight='bold'); c.set_height(c.get_height() * hm)
      elif j >= n:
         if m is not None and m.iat[i - 1, j - n]:
            c.set_facecolor(fc); c.get_text().set_color(tc)
         if bold is not None and bold.iat[i - 1, j - n]:
            c.get_text().set_weight('bold')
   fig.canvas.draw()
   r, inv = fig.canvas.get_renderer(), fig.dpi_scale_trans.inverted()
   tb, (w, h) = t.get_window_extent(r).transformed(inv), fig.get_size_inches()
   ttl = fig.text(tb.x0 / w, (tb.y1 + 0.1) / h, title, fontsize=12, fontweight='bold')
   fig.savefig(out + name, dpi=200, bbox_inches=Bbox.union([tb, ttl.get_window_extent(r).transformed(inv)]).padded(0.1))
   plt.close(fig)

def beta_mask(df):
   s = (df.xs('CI_lower', axis=1, level=1) < 0) & (df.xs('CI_upper', axis=1, level=1) > 0)
   m = s[df.columns.get_level_values(0)]; m.columns = df.columns; return m

def bold_mask(df):
   if isinstance(df.columns, pd.MultiIndex) and 'CI Relative Width Ratio' in df.columns.get_level_values(1):
      s = df.xs('CI Relative Width Ratio', axis=1, level=1) < 1.3
      m = s[df.columns.get_level_values(0)]; m.columns = df.columns; return m
   return None

for k, v in analysis_dfs.items():
   name = re.sub(r'\W+', '_', k).strip('_').lower() + '.png'
   is_beta = k == 'Beta & Confidence Intervals'
   title = v['title'] + ('  —  grey = 90% CI spans 0, bold = CI ratio < 1.3' if is_beta else '')
   save(v['df'], title, fmts.get(k, '{:.3f}'), name, (beta_mask(v['df']), '#d9d9d9', '#000000') if is_beta else None, bold_mask(v['df']) if is_beta else None)

vd, wald = analysis_dfs['Variance Decomposition']['df'], analysis_dfs['Wald Joint Tests']['df']
grouped = vd.groupby(level=0, sort=False).sum()
pv = wald.xs('p-value', axis=1, level=1).reindex(index=grouped.index, columns=grouped.columns)
save(grouped, 'Variance Decomposition by Channel (summed) — grey = Wald joint-test p-value > 0.10', '{:.2f}', 'variance_decomposition_grouped.png', (pv > 0.10, '#d9d9d9', '#000000'))

### Excel Output

In [11]:
def auto_fit_columns(ws, n_idx, min_row=4, padding=3):
   merged = {rc for rng in ws.merged_cells.ranges if rng.min_col != rng.max_col for rc in rng.cells}

   for col in ws.iter_cols(min_row=min_row):
      lengths = [len(f"{c.value:.4f}" if isinstance(c.value, (int, float)) else str(c.value))
                 for c in col if c.value is not None and (c.row, c.column) not in merged]
      width = max(lengths, default=10) + padding
      idx = col[0].column
      ws.column_dimensions[letter(idx)].width = max(width, 14) if idx <= n_idx else min(max(width, 10), 16)

def bands(index):
   s = pd.Series(index.get_level_values(0))
   return ((s != s.shift()).cumsum() % 2 == 1) & (index.nlevels > 1)

def add_banner(ws, row, text, font, span, height, wrap=False):
   cell = ws.cell(row, 1, text)
   cell.font = font
   cell.alignment = Alignment(None, "center", wrap_text=wrap)
   ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=span)
   ws.row_dimensions[row].height = height

with pd.ExcelWriter(f"{OUTPUT_FOLDER_PATH}/analysis_output.xlsx", "openpyxl") as writer:
   for sheet_name, content in analysis_dfs.items():
      df, title, caption = content["df"], content["title"], content["caption"]
      df.to_excel(writer, sheet_name=sheet_name, startrow=3)

      ws = writer.sheets[sheet_name]
      n_idx = df.index.nlevels
      n_cols = df.shape[1] + n_idx
      span = max(n_cols, 8)
      h = df.columns.nlevels
      first = h + 4 + (h > 1)     # first data row (MultiIndex columns add an index-names row)
      last = first + len(df) - 1  # last data row

      rows, cols = bands(df.index), bands(df.columns)

      for row in ws.iter_rows(4, last, 1, n_cols):
         for c in row:
            c.alignment = Alignment(c.alignment.horizontal, "center", wrap_text=c.alignment.wrap_text)
            i, j = c.row - first, c.column - n_idx - 1   # data-row / value-column position (negative in header / index)
            if (i >= 0 and rows[i]) | (j >= 0 and cols[j]):
               c.fill = PatternFill("solid", "F2F2F2")
            if i >= 0 and j >= 0 and isinstance(c.value, (int, float)):
               c.number_format = "0.0000"

      auto_fit_columns(ws, n_idx)
      for c in range(n_cols + 1, span + 1):
         ws.column_dimensions[letter(c)].width = 10

      add_banner(ws, 1, title, Font(sz=16, b=True, color="1F4E79"), span, 28)
      total = sum(ws.column_dimensions[letter(c)].width for c in range(1, span + 1))
      height = max(30, 18 * math.ceil(len(caption) / (total * 0.85)))
      add_banner(ws, 2, caption, Font(sz=10, i=True, color="595959"), span, height, True)